In [1]:
import collections
import random
import gymnasium as gym
import numpy as np
import pandas as pd

def sample_policy(state):
    player_score, dealer_card, usable_ace = state
    return 0 if player_score >= 20 else 1

def generate_prediction_episode(env, policy):
    episode = []
    state, info = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode.append((state, action, reward))
        state = next_state

    return episode

def first_visit_mc_prediction(env, policy, num_episodes):
    returns_sum = collections.defaultdict(float)
    returns_count = collections.defaultdict(float)
    V = collections.defaultdict(float)

    for i in range(num_episodes):
        episode = generate_prediction_episode(env, policy)
        G = 0
        visited_states = set()

        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G += reward

            if state not in visited_states:
                visited_states.add(state)
                returns_sum[state] += G
                returns_count[state] += 1.0
                V[state] = returns_sum[state] / returns_count[state]

    return V

def make_epsilon_greedy_policy(Q, epsilon, nA):
    def policy_fn(state):
        action_probs = np.ones(nA, dtype=float) * (epsilon / nA)
        best_action = np.argmax(Q[state])
        action_probs[best_action] += (1.0 - epsilon)
        return np.random.choice(np.arange(nA), p=action_probs)
    return policy_fn

def generate_control_episode(env, policy):
    episode = []
    state, info = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode.append((state, action, reward))
        state = next_state

    return episode

def mc_on_policy_control(env, num_episodes, epsilon=0.1):
    num_actions = env.action_space.n
    Q = collections.defaultdict(lambda: np.zeros(num_actions))
    returns_sum = collections.defaultdict(float)
    returns_count = collections.defaultdict(float)

    for i in range(num_episodes):
        current_policy = make_epsilon_greedy_policy(Q, epsilon, num_actions)
        episode = generate_control_episode(env, current_policy)
        G = 0
        visited_sa_pairs = set()

        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G += reward
            sa_pair = (state, action)

            if sa_pair not in visited_sa_pairs:
                visited_sa_pairs.add(sa_pair)
                returns_sum[sa_pair] += G
                returns_count[sa_pair] += 1.0
                Q[state][action] = returns_sum[sa_pair] / returns_count[sa_pair]

    return Q

if __name__ == "__main__":
    env = gym.make('Blackjack-v1', natural=False, sab=False)

    print("--- Running First-Visit MC Prediction (500,000 Episodes) ---")
    prediction_episodes = 500_000
    V_table = first_visit_mc_prediction(env, sample_policy, num_episodes=prediction_episodes)

    print("Sample Values from V(s) Table:")
    sample_states = [(20, 10, False), (13, 7, False), (21, 5, True)]
    for s in sample_states:
        print(f"  State {s} -> Value: {V_table[s]:.4f}")

    print("\n--- Running On-Policy MC Control (50,000 Episodes) ---")
    control_episodes = 50_000
    Q_table = mc_on_policy_control(env, num_episodes=control_episodes, epsilon=0.15)

    q_list = []
    for state, actions in Q_table.items():
        q_list.append({
            "Player Score": state[0],
            "Dealer Card": state[1],
            "Usable Ace": state[2],
            "Q(Stick)": actions[0],
            "Q(Hit)": actions[1],
            "Best Action": "Stick" if np.argmax(actions) == 0 else "Hit"
        })

    df_q = pd.DataFrame(q_list)

    print("\nFirst 10 rows of the Q-Values DataFrame:")
    print(df_q.head(10).to_string(index=False))

    env.close()


--- Running First-Visit MC Prediction (500,000 Episodes) ---
Sample Values from V(s) Table:
  State (20, 10, False) -> Value: 0.4369
  State (13, 7, False) -> Value: -0.5489
  State (21, 5, True) -> Value: 0.8942

--- Running On-Policy MC Control (50,000 Episodes) ---

First 10 rows of the Q-Values DataFrame:
 Player Score  Dealer Card  Usable Ace  Q(Stick)    Q(Hit) Best Action
           13            3           0 -0.233429 -0.274194       Stick
           12           10           0 -0.574468 -0.461740         Hit
           16           10           0 -0.628070 -0.615605         Hit
           21           10           1  0.882283  0.026316       Stick
           20           10           0  0.422307 -0.865169       Stick
           20            8           0  0.808511 -0.878788       Stick
           14            4           1 -0.333333  0.119048         Hit
           18            2           0  0.173729 -0.461538       Stick
           21           10           0  0.901333 -